# Mini-Project 1: Where Does GPT-2 Small "Know" Country Capitals?

**Timebox:** 1 to 2 days (throwaway mini-project: aim for fast iteration and real plots, not perfection).

**Core Question:** When GPT-2 Small completes `"The capital of France is" -> " Paris"`, at which layers does the answer emerge, and do **Attention heads** or **MLPs** write it to the residual stream?

**TransformerLens 4.0 primitives you'll practice:** `TransformerBridge.boot_transformers`, `model.to_single_token`, `model.to_str_tokens`, `utils.test_prompt`, `model.run_with_cache`, `cache.accumulated_resid`, `cache.decompose_resid`, `cache.logit_attrs`.

In [1]:
# Only install if missing (so re-running this cell is instant!)
import importlib.util, sys
if "google.colab" in sys.modules and importlib.util.find_spec("transformer_lens") is None:
    %pip install -q transformer-lens einops plotly

import torch
import einops
import plotly.express as px
import plotly.graph_objects as go
from transformer_lens.model_bridge import TransformerBridge
from transformer_lens import utilities as utils

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# TransformerLens 4.0 loads models via TransformerBridge
# enable_compatibility_mode() gives HookedTransformer-style W_U and folded LayerNorms
model = TransformerBridge.boot_transformers("gpt2", device=device)
model.enable_compatibility_mode()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using device: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## Step 1: Build & Inspect Your Dataset (Raw Data First!)

**TODO:**
1. Create a list of 25 to 30 `(country, capital)` pairs.
2. Check tokenization: filter to pairs where `" " + capital` is a **single token** in GPT-2 (`model.to_str_tokens(" " + capital, prepend_bos=False)`).
3. Run `utils.test_prompt` on a couple of examples, then check accuracy across all prompts of the form `f"The capital of {country} is"`.
4. Keep the prompts GPT-2 gets right (and note which ones it gets wrong, since surprising failures are worth inspecting!).

In [20]:
pairs = [
    ("France", "Paris"),
    # ("Germany", "Berlin"),
    # ("Japan", "Tokyo"),
    ("Italy", "Rome"),
    ("Spain", "Madrid"),
    ("Russia", "Moscow"),
    ("China", "Beijing"),
    ("Egypt", "Cairo"),
    ("Greece", "Athens"),
    ("Ireland", "Dublin"),
    ("Austria", "Vienna"),
    ("Poland", "Warsaw"),
    ("Portugal", "Lisbon"),
    ("Norway", "Oslo"),
    ("Sweden", "Stockholm"),
    ("Finland", "Helsinki"),
    ("Netherlands", "Amsterdam"),
    ("Belgium", "Brussels"),
    ("Hungary", "Budapest"),
    ("Romania", "Bucharest"),
    ("Czechia", "Prague"),
    ("Turkey", "Ankara"),
    ("Thailand", "Bangkok"),
    ("Vietnam", "Hanoi"),
    ("Indonesia", "Jakarta"),
    ("Philippines", "Manila"),
    ("South Korea", "Seoul"),
    ("India", "New Delhi"),
    ("Iraq", "Baghdad"),
    ("Iran", "Tehran"),
    ("Afghanistan", "Kabul"),
    ("Canada", "Ottawa"),
    ("Peru", "Lima"),
    ("Kenya", "Nairobi"),
    ("Australia", "Canberra"),
]

# utils.test_prompt("The capital of France is", " Paris", model)

# make the approved pairs (one token capital)
approved_pairs = []
for country, capital in pairs:
    capital_tokens = model.to_str_tokens(" " + capital, prepend_bos=False)
    if len(capital_tokens)==1:
        approved_pairs.append((country, capital))

# check whether they are the first prediction
high_prob_pairs = []
mistakes = []
for country, capital in approved_pairs:
    pair = (country, capital)
    logits = model(
        f'The capital of Japan is Tokyo. The capital of Germany is Berlin. The capital of {country} is'
        )
    last_logits = logits[0, -1] # batch=0, seq_idx=-1, vocab_size=all of them
    probs = torch.softmax(last_logits, dim=-1)

    capital_id = model.to_single_token(" "+capital) # get vocab id of capital

    # max_prob_id = last_logits.argmax()
    # if max_prob_id == capital_id:
    #     high_prob_pairs.append(pair)

    high_probs, high_prob_ids = probs.topk(k=1)
    if capital_id in high_prob_ids:
        high_prob_pairs.append(pair)
    else:
        mistaken_top_one = model.to_string(probs.argmax())
        mistakes.append((country, mistaken_top_one))

non_high_prob_pairs = [app_pair for app_pair in approved_pairs 
                       if app_pair not in high_prob_pairs]

print("Num approved pairs " + str(len(approved_pairs)))
print("Num high probability pairs " + str(len(high_prob_pairs)))
print("Non-high probability pairs" + "\n" + str(non_high_prob_pairs))
print("Mistakes" + "\n" + str(mistakes))

Num approved pairs 29
Num high probability pairs 25
Non-high probability pairs
[('China', 'Beijing'), ('Turkey', 'Ankara'), ('Canada', 'Ottawa'), ('Australia', 'Canberra')]
Mistakes
[('China', ' Shanghai'), ('Turkey', ' Istanbul'), ('Canada', ' Toronto'), ('Australia', ' Sydney')]


## Step 2: Logit Lens Across Layers

**TODO:**
1. Run `logits, cache = model.run_with_cache(prompts)` (or loop over prompts if they have different token lengths, grabbing the **final token position**).
2. Extract the accumulated residual stream after each layer (`cache.accumulated_resid(layer=-1, incl_mid=True, pos_slice=-1, return_labels=True)`).
3. Apply `cache.apply_ln_to_stack(...)` and project onto the unembedding direction for the correct capital (or logit difference: `correct_capital - wrong_capital`).
4. Plot the mean (and individual curves!) across layers. At which layer does the capital suddenly jump up?

In [3]:
# TODO: Implement Logit Lens across layers and plot


## Step 3: Direct Logit Attribution (Attention vs. MLP)

**TODO:**
1. Decompose the final residual stream into per-layer components: `embed`, `pos_embed`, `0_attn_out`, `0_mlp_out`, ..., `11_mlp_out` using `cache.decompose_resid(layer=-1, pos_slice=-1, return_labels=True)`.
2. Compute each component's direct contribution to the correct capital logit (or logit diff) using `cache.logit_attrs(...)` or manual `einops.einsum` with `model.W_U`.
3. Plot a bar chart of average contribution per component. Are **MLPs** or **Attention heads** doing the heavy lifting?
4. *(Stretch)* If attention matters at a specific layer, decompose down to individual attention heads (`cache.stack_head_results`).

In [4]:
# TODO: Implement Direct Logit Attribution (Attention vs MLP) and plot


## Step 4: Red-Team Your Finding!

Before believing your plot, try to **break** it:
1. **Rephrase the prompt:** Test at least 2 different templates:
   - `f"{country}'s capital city is"`
   - `f"In {country}, the capital is"`
   - Few-shot: `"France: Paris\nGermany: Berlin\nJapan:"`
2. Do the **same layers and MLPs/heads** dominate, or did your original plot mostly reflect the exact wording `"The capital of X is"`?
3. **Control / Baseline:** What if you measure `logit(correct_capital)` vs. `logit(correct_capital) - mean_logit(other_capitals)`? Does a layer just boost *all* city names instead of the *specific* capital?
4. Summarize what held up and what broke in `notes.md`.

In [5]:
# TODO: Run rephrasing & city-baseline red-team checks
